# HGP-clusterer : 4D Panoptic Segmentation sur SemanticKITTI

Ce notebook implémente un pipeline de segmentation panoptique 4D en utilisant **HGP-clusterer**.

**Pipeline :**
1.  **Setup** : Installation des dépendances (Geogram/CGAL, HGP-clusterer).
2.  **Data** : Chargement d'une séquence SemanticKITTI (depuis Google Drive).
3.  **Preprocessing** : Construction d'un nuage de points 4D (x, y, z, t) ou BEV-4D (x, y, t).
4.  **Clustering** : HGP-clusterer pour l'association spatio-temporelle avec choix de la fonction de splitting (Oracle ou Géométrique).
5.  **Evaluation** : Calcul de la métrique LSTQ (LiDAR Segmentation and Tracking Quality).
6.  **Visualisation** : Rendu 3D interactif.

In [ ]:
# @title 1.1 Choix du Backend Géométrique
# 'geogram' est recommandé pour la vitesse (headless). 'cgal' est plus lent mais exact.
BACKEND = 'cgal'  # @param ['geogram', 'cgal']
print(f"Backend sélectionné : {BACKEND}")

In [ ]:
%%bash
# @title 1.2 Installation des dépendances système
apt-get update -qq
apt-get install -y -qq build-essential cmake git libeigen3-dev libomp-dev

if [ "$BACKEND" = "cgal" ]; then
    apt-get install -y -qq libcgal-dev libtbb-dev libtbbmalloc2 libgmp-dev libmpfr-dev
fi

In [ ]:
# @title 1.3 Installation des dépendances Python
!pip install -q --upgrade pip setuptools wheel Cython cmake jedi gdown
!pip install -q numpy scipy scikit-learn plotly tqdm joblib open3d plyfile hdbscan pandas matplotlib pyyaml

In [ ]:
%%bash
# @title 1.4 Installation de HGP-clusterer et SemanticKITTI-API
set -euo pipefail
WORKDIR="/content"
mkdir -p "${WORKDIR}"
cd "${WORKDIR}"

# HGP-clusterer
if [ -d HGP-clusterer ]; then
    git -C HGP-clusterer pull --ff-only
else
    git clone https://github.com/Ludwig-H/HGP-clusterer.git
fi

# SemanticKITTI API (pour l'évaluation)
if [ -d semantic-kitti-api ]; then
    git -C semantic-kitti-api pull --ff-only
else
    git clone https://github.com/PRBonn/semantic-kitti-api.git
fi

In [ ]:
# @title 1.5 Compilation de HGP
import os
import sys

WORKDIR = "/content"
os.chdir(WORKDIR)

if BACKEND == 'geogram':
    if not os.path.exists('geogram'):
        print("Clonage de Geogram...")
        !git clone --recursive https://github.com/BrunoLevy/geogram.git
    
    print("Compilation de Geogram (Headless)...")
    !cmake -S geogram -B geogram/build -DCMAKE_BUILD_TYPE=Release -DGEOGRAM_WITH_GRAPHICS=OFF -DGEOGRAM_WITH_LUA=OFF -DGEOGRAM_WITH_GARGANTUA=OFF
    !cmake --build geogram/build --config Release --parallel 4
    !cmake --install geogram/build --prefix /usr/local
    os.environ['GEOGRAM_INSTALL_PREFIX'] = '/usr/local'

elif BACKEND == 'cgal':
    print("Configuration CGAL...")
    !python3 {WORKDIR}/HGP-clusterer/scripts/setup_cgal.py
    cgal_dir = f"{WORKDIR}/HGP-clusterer/CGALDelaunay"
    projects = [
        'EdgesCGALDelaunay2D', 'EdgesCGALDelaunay3D', 'EdgesCGALDelaunayND',
        'EdgesCGALWeightedDelaunay2D', 'EdgesCGALWeightedDelaunay3D', 'EdgesCGALWeightedDelaunayND'
    ]
    for proj in projects:
        p_path = f"{cgal_dir}/{proj}"
        !cmake -S {p_path} -B {p_path}/build -DCMAKE_BUILD_TYPE=Release
        !cmake --build {p_path}/build --config Release
        !cmake --install {p_path}/build --prefix {WORKDIR}/HGP-clusterer

os.chdir(f"{WORKDIR}/HGP-clusterer")
!rm -rf build dist *.egg-info
!pip install -v --no-deps .

os.environ["CGALDELAUNAY_ROOT"] = f"{WORKDIR}/HGP-clusterer/CGALDelaunay"

try:
    from hgp_clusterer import HGPClusterer
    print("✅ HGPClusterer installé.")
except ImportError as e:
    print(f"❌ Erreur import HGP: {e}")

In [ ]:
# @title 2.1 Configuration Séquence et Téléchargement
# IMPORTANT : Si vous ne voulez tester qu'une seule séquence, lancez cette cellule.
# Le téléchargement via gdown --folder récupère tout le dossier si on ne filtre pas.
# Ici, on télécharge tout le dataset SemanticKITTI (partiel) fourni via le lien Drive.

SEQUENCE_TO_TEST = 8 # @param {type:"integer"}
DOWNLOAD_DATA = True # @param {type:"boolean"}

# Mapping des IDs Google Drive des séquences individuelles.
# Remplissez ce dictionnaire si vous connaissez les IDs des sous-dossiers pour éviter de tout télécharger.
SEQUENCE_DRIVE_IDS = {
    # 8: "ID_SPECIFIQUE_SEQUENCE_08",
}

# Dossier Racine (contient toutes les séquences)
ROOT_FOLDER_ID = "1ORVzSo-TWbNHeAC0-k3mxX9AiJHI_tVu"

if DOWNLOAD_DATA:
    import os
    
    # Destination racine
    base_dest = "/content/semantic_kitti_data"
    
    if not os.path.exists(base_dest):
        print("Démarrage du téléchargement...")
        
        # Vérifie si on a un ID spécifique pour la séquence demandée
        seq_id = SEQUENCE_DRIVE_IDS.get(SEQUENCE_TO_TEST)
        
        if seq_id:
            print(f"Téléchargement de la séquence {SEQUENCE_TO_TEST} uniquement (ID: {seq_id})...")
            # On crée le dossier de la séquence pour gdown
            # Structure cible : /content/semantic_kitti_data/XX
            seq_str = f"{SEQUENCE_TO_TEST:02d}"
            target_dir = os.path.join(base_dest, seq_str)
            
            # Note: gdown --folder crée le dossier s'il n'existe pas, mais on veut s'assurer de la structure
            !gdown --folder {seq_id} -O {target_dir} --quiet --remaining-ok
            
        else:
            print(f"ID spécifique non trouvé pour la séquence {SEQUENCE_TO_TEST}.")
            print(f"Téléchargement du dataset complet depuis le dossier racine (ID: {ROOT_FOLDER_ID})...")
            print("Cela peut prendre du temps.")
            !gdown --folder {ROOT_FOLDER_ID} -O {base_dest} --quiet --remaining-ok
        
        print("Téléchargement terminé (ou limité par Google).")
    else:
        print(f"Dossier {base_dest} existe déjà. Skip download.")
else:
    print("Téléchargement désactivé.")

print(f"Séquence cible pour le test : {SEQUENCE_TO_TEST}")

In [ ]:
# @title 2.2 Loader SemanticKITTI
import os
import numpy as np
import glob

class SemanticKITTILoader:
    def __init__(self, base_path, sequence_num):
        self.seq_str = f"{sequence_num:02d}"
        
        # Recherche du dossier de la séquence. 
        # Structure attendue : base_path/08 ou base_path/sequences/08
        
        # 1. Chercher direct
        possible_paths = glob.glob(f"{base_path}/{self.seq_str}")
        
        # 2. Chercher dans un sous-dossier 'sequences' (structure officielle KITTI)
        if not possible_paths:
            possible_paths = glob.glob(f"{base_path}/**/sequences/{self.seq_str}", recursive=True)
            
        # 3. Chercher récursivement n'importe où (au cas où gdown a créé une structure intermédiaire)
        if not possible_paths:
             possible_paths = glob.glob(f"{base_path}/**/{self.seq_str}", recursive=True)

        # Filtrer pour ne garder que les vrais dossiers contenant 'velodyne'
        valid_paths = []
        for p in possible_paths:
            if os.path.exists(os.path.join(p, 'velodyne')):
                valid_paths.append(p)
        
        if not valid_paths:
            raise ValueError(f"Séquence {self.seq_str} introuvable dans {base_path}. Vérifiez que le dossier 'velodyne' est bien présent.")
            
        self.seq_path = valid_paths[0]
        print(f"Séquence chargée : {self.seq_path}")
        
        self.velo_path = os.path.join(self.seq_path, 'velodyne')
        self.label_path = os.path.join(self.seq_path, 'labels')
        self.poses_file = os.path.join(self.seq_path, 'poses.txt')
        self.calib_file = os.path.join(self.seq_path, 'calib.txt')
        
        # Fallback pour poses.txt/calib.txt s'ils sont dans le dossier parent (structure dataset/sequences/08)
        if not os.path.exists(self.poses_file):
             # Essayer de remonter d'un niveau (dataset/sequences/) ou deux
             parent = os.path.dirname(self.seq_path) # dataset/sequences
             grandparent = os.path.dirname(parent) # dataset
             
             # Cas dataset/poses.txt (peu probable mais...)
             # Cas dataset/sequences/08/poses.txt (standard)
             pass 

        self.scan_files = sorted(glob.glob(os.path.join(self.velo_path, '*.bin')))
        self.label_files = sorted(glob.glob(os.path.join(self.label_path, '*.label')))
        self.poses = self._load_poses()
        self.calib = self._load_calib()
        
    def _load_poses(self):
        if not os.path.exists(self.poses_file):
            print(f"Info: poses.txt non trouvé ({self.poses_file}).")
            return []
        poses = []
        with open(self.poses_file, 'r') as f:
            for line in f:
                values = [float(v) for v in line.strip().split()]
                pose = np.vstack([np.array(values).reshape(3, 4), [0, 0, 0, 1]])
                poses.append(pose)
        return poses

    def _load_calib(self):
        if not os.path.exists(self.calib_file): 
            print(f"Info: calib.txt non trouvé ({self.calib_file}).")
            return np.eye(4)
        calib = {}
        with open(self.calib_file, 'r') as f:
            for line in f:
                if ':' not in line: continue
                key, val = line.split(':', 1)
                calib[key] = np.array([float(x) for x in val.split()]).reshape(3, 4)
        if 'Tr' in calib:
            return np.vstack([calib['Tr'], [0, 0, 0, 1]])
        return np.eye(4)

    def get_scan(self, idx, apply_pose=True):
        scan = np.fromfile(self.scan_files[idx], dtype=np.float32).reshape(-1, 4)
        points = scan[:, :3]
        if apply_pose and self.poses and idx < len(self.poses):
            T = self.poses[idx] @ self.calib
            points = (T @ np.hstack([points, np.ones((len(points), 1))]).T).T[:, :3]
        return points

    def get_labels(self, idx):
        if idx >= len(self.label_files): return None, None
        label = np.fromfile(self.label_files[idx], dtype=np.uint32)
        return label & 0xFFFF, label >> 16

    def __len__(self): return len(self.scan_files)

In [ ]:
# @title 3.1 Construction du Nuage 4D
import numpy as np

START_FRAME = 0 # @param {type:"integer"}
NUM_FRAMES = 10 # @param {type:"integer"}
DT_SCALE = 5.0  # @param {type:"number"}
APPLY_BEV = True # @param {type:"boolean"}

# Initialisation des variables pour éviter les NameError
X_clustering = None
X_4d = None
Y_sem = None
Y_inst = None
Time_idx = None

try:
    loader = SemanticKITTILoader("/content/semantic_kitti_data", SEQUENCE_TO_TEST)
    points_4d, gt_sem, gt_inst, times = [], [], [], []
    
    print(f"Chargement frames {START_FRAME} -> {START_FRAME + NUM_FRAMES}...")
    for i in range(NUM_FRAMES):
        idx = START_FRAME + i
        if idx >= len(loader): break
        
        pts = loader.get_scan(idx, apply_pose=True)
        s, inst = loader.get_labels(idx)
        
        # 4D Point: x, y, z, t
        t_col = np.full((len(pts), 1), i * DT_SCALE)
        points_4d.append(np.hstack([pts, t_col]))
        gt_sem.append(s)
        gt_inst.append(inst)
        times.extend([i] * len(pts))
    
    if points_4d:
        X_4d = np.vstack(points_4d)
        Y_sem = np.hstack(gt_sem)
        Y_inst = np.hstack(gt_inst)
        Time_idx = np.array(times)
        
        # Bird's Eye View : on utilise uniquement x, y, t pour le clustering
        if APPLY_BEV:
            print("Mode Bird's-Eye-View (BEV) activé : Clustering sur (x, y, t).")
            X_clustering = np.column_stack([X_4d[:, 0], X_4d[:, 1], X_4d[:, 3]])
        else:
            print("Mode 4D Complet activé : Clustering sur (x, y, z, t).")
            X_clustering = X_4d
        
        print(f"Nuage 4D: {X_4d.shape} points.")
        print(f"Input Clustering: {X_clustering.shape}")
    else:
        print("Aucun point chargé. Vérifiez les chemins.")

except Exception as e:
    print(f"Erreur lors du chargement des données: {e}")
    print("---------------------------------------------------------")
    print("⚠️ GÉNÉRATION DE DONNÉES SYNTHÉTIQUES (FALLBACK) ⚠️")
    print("---------------------------------------------------------")
    # Fallback : Génération de données synthétiques pour que le notebook puisse continuer
    from sklearn.datasets import make_blobs
    n_samples = 5000
    X_syn, y_syn = make_blobs(n_samples=n_samples, n_features=3, centers=5, cluster_std=1.0)
    # Ajout dimension temps synthétique
    t_syn = np.random.randint(0, NUM_FRAMES, size=n_samples) * DT_SCALE
    X_4d = np.column_stack([X_syn, t_syn])
    Y_sem = np.zeros(n_samples, dtype=int)
    Y_inst = y_syn + 1 # Instance ID > 0
    Time_idx = (t_syn / DT_SCALE).astype(int)
    X_clustering = X_4d if not APPLY_BEV else X_4d[:, [0, 1, 3]]
    print(f"Données synthétiques générées: {X_clustering.shape}")

In [ ]:
# @title 4.1 HGP Clustering
import time
import numpy as np

# --- Import sécurisé de HGPClusterer ---
try:
    from hgp_clusterer import HGPClusterer
except ImportError:
    print("⚠️ Module HGPClusterer introuvable. Tentative de correction du path...")
    import sys
    if "/content/HGP-clusterer" not in sys.path:
        sys.path.append("/content/HGP-clusterer")
    try:
        from hgp_clusterer import HGPClusterer
        print("✅ HGPClusterer importé avec succès après correction du path.")
    except ImportError as e:
        raise RuntimeError(f"❌ Impossible d'importer HGPClusterer même après correction. Erreur: {e}. Veuillez vérifier la compilation en section 1.5.")

K = 5 # @param {type:"integer"}
MIN_CLUSTER_SIZE = 50 # @param {type:"integer"}
SPLIT_MODE = "Oracle" # @param ["None", "Oracle", "Geometric"]

# Vérification préalable des données
if X_clustering is None:
    raise RuntimeError("Erreur critique: X_clustering n'est pas défini. Veuillez vérifier la cellule 'Construction du Nuage 4D'.")

# --- Fonctions de Splitting ---

def oracle_split(parent, children):
    "Split si les enfants sont nettement plus purs sémantiquement que le parent."
    # Utilise Y_inst global
    p_labels = Y_inst[parent]; p_labels = p_labels[p_labels > 0]
    if len(p_labels) == 0: return False
    
    # Pureté majoritaire du parent
    parent_purity = np.max(np.unique(p_labels, return_counts=True)[1]) / len(p_labels)
    
    child_purity_sum = 0; total = 0
    for c in children:
        c_labels = Y_inst[c]; c_labels = c_labels[c_labels > 0]
        if len(c_labels) > 0:
            child_purity_sum += np.max(np.unique(c_labels, return_counts=True)[1])
            total += len(c_labels)
    
    child_purity = child_purity_sum / total if total > 0 else 0
    
    # Critère : On split si on gagne en pureté
    return child_purity > parent_purity + 0.05

def geometric_split(parent, children):
    "Split si le parent est géométriquement incohérent (variance trop élevée) comparé aux enfants."
    pts_parent = X_clustering[parent]
    
    # Calcul variance spatiale (x, y uniquement pour robustesse)
    var_parent = np.var(pts_parent[:, :2], axis=0).sum()
    
    weighted_var_children = 0
    total_len = 0
    
    for c in children:
        pts_child = X_clustering[c]
        if len(pts_child) < 2: continue
        v = np.var(pts_child[:, :2], axis=0).sum()
        weighted_var_children += v * len(pts_child)
        total_len += len(pts_child)
        
    avg_var_children = weighted_var_children / total_len if total_len > 0 else var_parent
    
    # Critère : Si la variance du père est significativement plus grande que la moyenne des fils
    # Cela suggère que le père regroupe des clusters bien séparés spatialement
    if var_parent > 1.5 * avg_var_children and var_parent > 5.0: # Seuil arbitraire 5m^2
        return True
    return False

# --- Sélection de la fonction ---
if SPLIT_MODE == "Oracle":
    split_func = oracle_split
    print("Fonction de splitting : Oracle (basée sur la vérité terrain)")
elif SPLIT_MODE == "Geometric":
    split_func = geometric_split
    print("Fonction de splitting : Géométrique (basée sur la variance spatiale)")
else:
    split_func = None
    print("Fonction de splitting : Aucune (Clustering standard)")

clusterer = HGPClusterer(
    K=K, min_cluster_size=MIN_CLUSTER_SIZE, min_samples=K+1,
    splitting=split_func,
    cgal_root=os.environ.get("CGALDELAUNAY_ROOT"),
    verbose=True
)

print("Clustering en cours...")
t0 = time.time()
try:
    labels_pred = clusterer.fit_predict(X_clustering)
    print(f"Fini en {time.time()-t0:.2f}s. Clusters trouvés : {len(set(labels_pred))-1}")
except Exception as e:
    print(f"Erreur durant le clustering: {e}")
    # Fallback labels pour que la suite ne plante pas
    labels_pred = np.zeros(len(X_clustering), dtype=int)

In [ ]:
# @title 5.1 Évaluation LSTQ (Panoptic)
# Cette cellule calcule une approximation du LSTQ (LiDAR Segmentation and Tracking Quality)
# S_assoc : Association Quality (Tracking IoU)
# S_cls   : Classification Quality (Semantic IoU - ici on suppose la sémantique connue/oracle pour l'instant)

def compute_lstq_simplified(pred_labels, gt_inst_labels, gt_sem_labels):
    """
    Calcul simplifié de S_assoc.
    On associe chaque cluster prédit à l'instance GT majoritaire.
    """
    # Ignorer le bruit
    mask = (gt_inst_labels > 0) & (pred_labels >= 0)
    if np.sum(mask) == 0: return 0.0, 0.0
    
    # Intersection Matrix: rows=GT, cols=Pred
    # C'est une approximation, le vrai LSTQ utilise une association hongroise sur les tubes 4D
    
    from sklearn.metrics import confusion_matrix
    
    # Remap labels to 0..N for confusion matrix
    u_gt, inv_gt = np.unique(gt_inst_labels[mask], return_inverse=True)
    u_pred, inv_pred = np.unique(pred_labels[mask], return_inverse=True)
    
    cm = confusion_matrix(inv_gt, inv_pred)
    
    # T-IoU pour chaque paire (GT_i, Pred_j)
    # IoU = Intersection / Union
    # Intersection = cm[i, j]
    # Union = count_gt[i] + count_pred[j] - cm[i, j]
    
    count_gt = cm.sum(axis=1)
    count_pred = cm.sum(axis=0)
    
    # Association greedy : Pour chaque GT, on prend le Pred qui maximise l'IoU
    ious = []
    for i in range(len(u_gt)):
        best_iou = 0
        for j in range(len(u_pred)):
            inter = cm[i, j]
            union = count_gt[i] + count_pred[j] - inter
            if union > 0:
                iou = inter / union
                if iou > best_iou: best_iou = iou
        ious.append(best_iou)
        
    s_assoc = np.mean(ious) if ious else 0.0
    return s_assoc

if X_clustering is not None and len(X_clustering) > 0:
    print("Calcul du LSTQ (Approximation S_assoc)...")
    s_assoc = compute_lstq_simplified(labels_pred, Y_inst, Y_sem)
    print(f"S_assoc (Tracking Quality) : {s_assoc:.4f}")
    print("Note: Ceci est une approximation. Pour le score officiel, utilisez evaluate_panoptic.py de l'API SemanticKITTI.")
else:
    print("Pas de données pour l'évaluation.")

In [ ]:
# @title 6.1 Visualisation 3D
import plotly.graph_objects as go

if X_4d is not None and len(X_4d) > 0:
    idx = np.arange(0, len(X_4d), 10) # Downsample
    X_v = X_4d[idx]
    L_v = labels_pred[idx]
    
    fig = go.Figure()
    u_l = np.unique(L_v)
    
    for l in u_l:
        if l == -1: continue
        m = L_v == l
        fig.add_trace(go.Scatter3d(
            x=X_v[m, 0], y=X_v[m, 1], z=X_v[m, 2],
            mode='markers', marker=dict(size=2), name=f'C{l}'
        ))
    fig.update_layout(title="HGP Clusters 4D", scene=dict(aspectmode='data'))
    fig.show()
else:
    print("Pas de données à afficher.")